In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
import seaborn as sns
import sys
sys.path.append("/home/jg2447/slayman/motif_inference/MOBI/scripts/")
from src import plt_func
from src import encode_meta

### data

In [ ]:
# Get Cell Line Specific TFs and common TFs
metadata = encode_meta.unique_TF_parser("/home/jg2447/slayman/motif_inference/result/metadata/human-GM12878.txt")
u_TF_files, u_TF_names, u_TF_files_w_motif, u_TF_names_w_motif, u_motif_files = metadata[5:]
TF_GM12878 = u_TF_names_w_motif

metadata = encode_meta.unique_TF_parser("/home/jg2447/slayman/motif_inference/result/metadata/human-K562.txt")
u_TF_files, u_TF_names, u_TF_files_w_motif, u_TF_names_w_motif, u_motif_files = metadata[5:]
TF_K562 = u_TF_names_w_motif

TF_common = set.intersection(set(TF_GM12878), set(TF_K562))
TF_common = np.array(list(TF_common))
TF_common.sort()

# gene names annotations
tf_name2id = pd.read_csv("/home/jg2447/slayman/data/Ensembl_biomart/Ensembl_hg19_geneID_geneIDver_geneName_chr.txt", sep="\t", header=None)
tf_name2tid = pd.read_csv("/home/jg2447/slayman/data/Ensembl_biomart/Ensembl_hg19_transcriptID_transcriptIDver_geneName_chr.txt", sep="\t", header=None)
tf_name2id_common = tf_name2id[tf_name2id[2].isin(TF_common)]

# read RNA-seq data 
# normalize so that top5 genes having same mean
def tpm_input(f, factor=100000):
    df = pd.read_csv(f, sep="\t")
    df = df.sort_values("TPM", ascending=False)
    coef = df.iloc[:5,:]['TPM'].mean() / factor
    df["TPM_new"] = df["TPM"] / coef
    return(df)

K562 = tpm_input("/home/jg2447/slayman/data/ENCODE_total_RNA_seq/K562_RNA-seq_gene_rep1_ENCFF139IXQ.tsv")
K562 = pd.merge(K562, tf_name2id, left_on="gene_id", right_on=1)
GM12878 = tpm_input("/home/jg2447/slayman/data/ENCODE_total_RNA_seq/GM12878_RNA-seq_gene_rep1_ENCFF009ZXH.tsv")
GM12878 = pd.merge(GM12878, tf_name2id, left_on="gene_id", right_on=1)

# original TPM of only common TFs
TPM_GM12878 = GM12878[GM12878['gene_id'].isin(tf_name2id_common[1])].sort_values(2)["TPM"].values
TPM_K562 = K562[K562['gene_id'].isin(tf_name2id_common[1])].sort_values(2)["TPM"].values
# normalized TPM of only common TFs
TPM_new_GM12878 = GM12878[GM12878['gene_id'].isin(tf_name2id_common[1])].sort_values(2)["TPM_new"].values
TPM_new_K562 = K562[K562['gene_id'].isin(tf_name2id_common[1])].sort_values(2)["TPM_new"].values

# enrichment data of common TF
data = pd.read_csv("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/humanGM12878/enrichment/RankSPP_100.txt", sep="\t", index_col=0).sort_index()
enrichment_SPP_GM12878 = data[data.index.isin(TF_common)]["250"].values
data = pd.read_csv("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/humanK562/enrichment/RankSPP_100.txt", sep="\t", index_col=0).sort_index()
enrichment_SPP_K562 = data[data.index.isin(TF_common)]["250"].values

# inference data of common TF
data = pd.read_csv("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/humanGM12878/tomtom_summary/DREME_RankSPP_100_top5.txt", sep="\t")
accuracy_SPP_GM12878 = data[data["TF"].isin(TF_common)].sort_values("TF")["accuracy"].values
data = pd.read_csv("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/humanK562/tomtom_summary/DREME_RankSPP_100_top5.txt", sep="\t")
accuracy_SPP_K562 = data[data["TF"].isin(TF_common)].sort_values("TF")["accuracy"].values
data = pd.read_csv("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/humanGM12878/tomtom_summary/DREME_RankLinear_1.0_100_top5.txt", sep="\t")
accuracy_MOBI_GM12878 = data[data["TF"].isin(TF_common)].sort_values("TF")["accuracy"].values
data = pd.read_csv("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/humanK562/tomtom_summary/DREME_RankLinear_1.0_100_top5.txt", sep="\t")
accuracy_MOBI_K562 = data[data["TF"].isin(TF_common)].sort_values("TF")["accuracy"].values

# combine all data
data = pd.DataFrame([TPM_new_GM12878, TPM_new_K562, enrichment_SPP_GM12878, enrichment_SPP_K562, accuracy_SPP_GM12878, accuracy_SPP_K562, accuracy_MOBI_GM12878, accuracy_MOBI_K562]).T
data.index = TF_common
data.columns = ["exp_normal_GM12878", "exp_normal_K562", "enrichment_SPP_GM12878", "enrichment_SPP_K562", "inference_SPP_GM12878", "inference_SPP_K562", "inference_MOBI_GM12878", "inference_MOBI_K562"]

# divide all data into two half base on expression ratio
data["ratio"] = data["exp_normal_K562"]/data["exp_normal_GM12878"].fillna(0)
data = data.sort_values("ratio", ascending=False)
data.to_csv("./fig5_data.txt", sep="\t")

### plot

In [ ]:
def plt_bar(ax, data, data_err):
    ##
    ax.bar(
        [0,0.45],
        data, 
        width=0.4,
        color=colors, linewidth=0.3, edgecolor="black")

    _, caps, sticks = ax.errorbar(
        x=[0,0.45],
        y=data,
        yerr=data_err,
        fmt=".", lw=1, capsize=1, markersize=0, elinewidth=0.5, ecolor=(0.1, 0.1, 0.1, 0.75))
    for cap in caps:
         cap.set_markeredgewidth(0.5)

    ax.set_xlim([-0.45, 0.9])
    ax.get_xaxis().set_ticks([])
    ax.get_xaxis().set_ticklabels([])
    ax.get_xaxis().set_tick_params(size=0, pad=2, labelsize=x_tick_label_fs)

    # ax.get_yaxis().set_ticks([0,0.5,1,1.5,2])
    ax.get_yaxis().set_tick_params(size=2.5, width=0.5, pad=1, labelsize=y_tick_label_fs)

    #ax.legend([ax.patches[0], ax.patches[1]], ["GM12878", "K562"], fontsize=legend_fs, loc='upper center', ncol=2, frameon=False, bbox_to_anchor=(0.4, 1.15))

    plt_func.ax_trim(ax)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines["left"].set_linewidth(0.5)


In [ ]:
figsize = (3.2, 2.8)
panel_number_fs = 8
x_tick_label_fs = 5
y_tick_label_fs = 5
x_label_fs = 6
y_label_fs = 6
title_fs = 5
legend_fs = 5

colors = ["#5975a4", "#5f9e6e"]

sns.set_context("paper")

fig, axs = plt.subplots(
    nrows=2, ncols=1,
    gridspec_kw={'hspace': 0.55},
    figsize=figsize, dpi=300)

axs[0].axis('off') # A-B
axs[1].axis('off') # C-D

# read data
data = pd.read_csv("./fig5_data.txt", sep="\t", index_col=0)
highK = data.iloc[:data.shape[0]//2,]
lowK = data.iloc[data.shape[0]//2:,]

################ A-B
gs = mpl.gridspec.GridSpecFromSubplotSpec(1, 2, subplot_spec=axs[0], wspace=0.3)

## A
ax = plt.subplot(gs[0,0])
ax.plot(data["enrichment_SPP_GM12878"], data["enrichment_SPP_K562"], "o", ms=3, markeredgewidth=0.3, markeredgecolor='k', color="#b55d60")
ax.plot([-0.1,1.1], [-0.1,1.1], "k--", lw=1)
ax.set_xticks([0,0.5,1])
ax.set_yticks([0,0.5,1])
ax.set_xlabel("GM12878", size=x_label_fs, labelpad=1)
ax.set_ylabel("K562", size=y_label_fs, labelpad=1)
ax.get_xaxis().set_tick_params(size=2, width=0.5, labelsize=x_tick_label_fs, pad=1)
ax.get_yaxis().set_tick_params(size=2, width=0.5, labelsize=y_tick_label_fs, pad=1)
ax.set_title("Motif enrichment", size=title_fs, pad=0)
plt_func.ax_trim(ax)

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines["left"].set_linewidth(0.5)
ax.spines["bottom"].set_linewidth(0.5)

ax.text(-0.10, 1.07, "A", transform=ax.transAxes, size=panel_number_fs, weight='bold')
ax.text(0.6, -0.1, "p = 0.06", fontsize=5)
    
## B
ax = plt.subplot(gs[0,1])
ax.bar(
    [0,0.45,1.3,1.75],
    [data["inference_SPP_GM12878"].mean(), data["inference_SPP_K562"].mean(), data["inference_MOBI_GM12878"].mean(), data["inference_MOBI_K562"].mean()],
    width=0.4,
    color=[colors[0], colors[1], colors[0], colors[1]], linewidth=0.3, edgecolor="black")

_, caps, sticks = ax.errorbar(
    x=[0,0.45,1.3,1.75],
    y=[data["inference_SPP_GM12878"].mean(), data["inference_SPP_K562"].mean(), data["inference_MOBI_GM12878"].mean(), data["inference_MOBI_K562"].mean()],
    yerr=[data["inference_SPP_GM12878"].sem(), data["inference_SPP_K562"].sem(), data["inference_MOBI_GM12878"].sem(), data["inference_MOBI_K562"].sem()],
    fmt=".", lw=1, capsize=1, markersize=0, elinewidth=0.5, ecolor=(0.1, 0.1, 0.1, 0.75))
for cap in caps:
     cap.set_markeredgewidth(0.5)

ax.set_xlim([-0.45, 3])
ax.get_xaxis().set_ticks([0.225, 1.525])
ax.get_xaxis().set_ticklabels(["SPP", "MOBI"])
ax.get_xaxis().set_tick_params(size=0, pad=2, labelsize=x_tick_label_fs)

ax.get_yaxis().set_ticks([0,0.5,1,1.5,2])
ax.get_yaxis().set_tick_params(size=2.5, width=0.5, pad=3, labelsize=y_tick_label_fs)
ax.set_ylabel("Correct inference", fontsize=y_label_fs, labelpad=1)

ax.legend(
    [ax.patches[0], ax.patches[1]],
    ["GM12878", "K562"],
    fontsize=legend_fs-0.5, loc='upper center', ncol=2, frameon=False, bbox_to_anchor=(0.4, 1.2), handlelength=1.5)

plt_func.ax_trim(ax)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines["left"].set_linewidth(0.5)

ax.text(-0.18, 1.1, "B", transform=ax.transAxes, size=panel_number_fs, weight='bold')

################ C-D
gs = mpl.gridspec.GridSpecFromSubplotSpec(1, 4, subplot_spec=axs[1], wspace=0.6)

ax = plt.subplot(gs[0,0]) # C-left
plt_bar(ax, [highK['enrichment_SPP_GM12878'].mean(), highK['enrichment_SPP_K562'].mean()], [highK['enrichment_SPP_GM12878'].sem(), highK['enrichment_SPP_K562'].sem()])
ax.set_ylabel("Motif enrichment", fontsize=y_label_fs, labelpad=1)
ax.text(-0.18, 1.1, "C", transform=ax.transAxes, size=panel_number_fs, weight='bold')

ax = plt.subplot(gs[0,1]) # C-right
plt_bar(ax, [highK['inference_SPP_GM12878'].mean(), highK['inference_SPP_K562'].mean()], [highK['inference_SPP_GM12878'].sem(), highK['inference_SPP_K562'].sem()])
ax.set_ylabel("Correct inference", fontsize=y_label_fs, labelpad=1)
ax.set_title("TFs that tend to \nexpress higher in K562", fontsize=title_fs, x=-0.3, y=1)

ax = plt.subplot(gs[0,2]) # D-left
plt_bar(ax, [lowK['enrichment_SPP_GM12878'].mean(), lowK['enrichment_SPP_K562'].mean()], [lowK['enrichment_SPP_GM12878'].sem(), lowK['enrichment_SPP_K562'].sem()])
ax.set_ylabel("Motif enrichment", fontsize=y_label_fs, labelpad=1)
ax.text(-0.18, 1.1, "D", transform=ax.transAxes, size=panel_number_fs, weight='bold')

ax = plt.subplot(gs[0,3]) # D-right
plt_bar(ax, [lowK['inference_SPP_GM12878'].mean(), lowK['inference_SPP_K562'].mean()], [lowK['inference_SPP_GM12878'].sem(), lowK['inference_SPP_K562'].sem()])
ax.set_ylabel("Correct inference", fontsize=y_label_fs, labelpad=1)
ax.set_title("TFs that tend to \nexpress lower in K562", fontsize=title_fs, x=-0.3, y=1)

# plt.show()
plt.savefig("./fig5.pdf", dpi="figure", bbox_inches="tight")